In [0]:

%pip install -q xgboost lightgbm optuna

In [0]:
# =========================
# CONFIGURATION
# =========================
dbutils.widgets.text("TABLE_NAME", "ml.data.model_24h")
dbutils.widgets.text("TARGET_COL", "label_fail_24h")
dbutils.widgets.text("HORIZON_HOURS", "24")
dbutils.widgets.text("MODEL_NAME_PREFIX", "wind_turbine_failure_turned")
dbutils.widgets.text("CATALOG_NAME", "ml")
dbutils.widgets.text("SCHEMA_NAME", "models")
dbutils.widgets.text("EXPERIMENT_NAME", "/Shared/wind_turbine_failure_turned")

In [0]:
# =========================
# INSTALL DEPENDENCIES
# =========================
%pip install -q xgboost lightgbm optuna




# =========================
# IMPORTS
# =========================
import pyspark.sql.functions as F
import mlflow
import optuna
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve
)

import xgboost as xgb
from lightgbm import LGBMClassifier
from mlflow.models.signature import infer_signature

# =========================
# MLFLOW SETUP
# =========================
def setup_mlflow(experiment_name: str):
    mlflow.set_experiment(experiment_name)
    mlflow.set_registry_uri("databricks-uc")

# =========================
# DATA UTILITIES
# =========================
def load_data(table: str):
    return spark.table(table)

def get_feature_columns(df, target: str):
    return [c for c in df.columns if c not in [target, "hour_start", "turbine_id"]]

def time_split(df):
    train = df.filter("hour_start <= '2023-06-30'")
    test  = df.filter("hour_start >= '2023-07-01'")
    return train, test

def compute_scale_pos_weight(df, target: str):
    neg = df.filter(F.col(target) == 0).count()
    pos = df.filter(F.col(target) == 1).count()
    return neg / max(pos, 1)

def prepare_pandas(train_df, test_df, features, target):
    X_train = train_df.select(features).toPandas()
    y_train = train_df.select(target).toPandas().values.ravel()
    X_test  = test_df.select(features).toPandas()
    y_test  = test_df.select(target).toPandas().values.ravel()
    return X_train, y_train, X_test, y_test

# =========================
# METRICS
# =========================
def find_best_threshold(y_true, y_prob):
    p, r, t = precision_recall_curve(y_true, y_prob)
    f1 = 2 * (p * r) / (p + r + 1e-9)
    idx = np.argmax(f1)
    return t[idx], f1[idx]

# =========================
# OPTUNA OBJECTIVES
# =========================
def xgb_objective(trial, X_train, y_train, X_test, y_test, scale_pos_weight):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 500),
        "max_depth": trial.suggest_int("max_depth", 4, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
        "scale_pos_weight": scale_pos_weight,
        "eval_metric": "aucpr",
        "random_state": 42,
        "n_jobs": -1
    }

    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train, verbose=False)

    probs = model.predict_proba(X_test)[:, 1]
    _, f1 = find_best_threshold(y_test, probs)
    return f1


def lgbm_objective(trial, X_train, y_train, X_test, y_test, scale_pos_weight):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 500),
        "num_leaves": trial.suggest_int("num_leaves", 31, 128),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
        "scale_pos_weight": scale_pos_weight,
        "random_state": 42,
        "n_jobs": -1
    }

    model = LGBMClassifier(**params)
    model.fit(X_train, y_train)

    probs = model.predict_proba(X_test)[:, 1]
    _, f1 = find_best_threshold(y_test, probs)
    return f1

# =========================
# HYPERPARAMETER TUNING
# =========================
def run_optuna(X_train, y_train, X_test, y_test, scale_pos_weight, n_trials=5):
    studies = {
        "xgboost": optuna.create_study(direction="maximize"),
        "lightgbm": optuna.create_study(direction="maximize")
    }

    studies["xgboost"].optimize(
        lambda t: xgb_objective(t, X_train, y_train, X_test, y_test, scale_pos_weight),
        n_trials=n_trials
    )

    studies["lightgbm"].optimize(
        lambda t: lgbm_objective(t, X_train, y_train, X_test, y_test, scale_pos_weight),
        n_trials=n_trials
    )

    return studies

# =========================
# TRAIN FINAL MODELS + TRACK
# =========================
def train_log_select_best(studies, X_train, y_train, X_test, y_test):
    best_global = {"f1": -1}

    for model_type, study in studies.items():
        with mlflow.start_run(run_name=f"{model_type}_optuna_best") as run:
            params = study.best_params
            params["random_state"] = 42
            params["n_jobs"] = -1

            if model_type == "xgboost":
                params["eval_metric"] = "aucpr"
                model = xgb.XGBClassifier(**params)
            else:
                model = LGBMClassifier(**params)

            model.fit(X_train, y_train)
            probs = model.predict_proba(X_test)[:, 1]

            thr, f1 = find_best_threshold(y_test, probs)
            preds = (probs >= thr).astype(int)

            mlflow.log_metrics({
                "pr_auc": average_precision_score(y_test, probs),
                "precision": precision_score(y_test, preds),
                "recall": recall_score(y_test, preds),
                "f1": f1,
                "threshold": thr
            })

            signature = infer_signature(X_train, model.predict(X_train))
            input_example = X_train.iloc[[0]]

            mlflow.sklearn.log_model(
                model,
                "model",
                signature=signature,
                input_example=input_example
            )

            if f1 > best_global["f1"]:
                best_global = {
                    "f1": f1,
                    "run_id": run.info.run_id
                }

    return best_global

# =========================
# REGISTER MODEL
# =========================
def register_best_model(best, catalog, schema, prefix, horizon):
    model_name = f"{catalog}.{schema}.{prefix}_{horizon}h"
    model_uri = f"runs:/{best['run_id']}/model"

    mlflow.register_model(model_uri=model_uri, name=model_name)

    print("✅ BEST MODEL REGISTERED")
    print("Model:", model_name)
    print("F1:", best["f1"])

# =========================
# MAIN PIPELINE
# =========================
def main():
    table   = dbutils.widgets.get("TABLE_NAME")
    target  = dbutils.widgets.get("TARGET_COL")
    horizon = dbutils.widgets.get("HORIZON_HOURS")
    prefix  = dbutils.widgets.get("MODEL_NAME_PREFIX")
    catalog = dbutils.widgets.get("CATALOG_NAME")
    schema  = dbutils.widgets.get("SCHEMA_NAME")
    experiment = dbutils.widgets.get("EXPERIMENT_NAME")

    setup_mlflow(experiment)

    df = load_data(table)
    features = get_feature_columns(df, target)

    train_df, test_df = time_split(df)
    scale_pos_weight = compute_scale_pos_weight(train_df, target)

    X_train, y_train, X_test, y_test = prepare_pandas(
        train_df, test_df, features, target
    )

    studies = run_optuna(
        X_train, y_train, X_test, y_test, scale_pos_weight
    )

    best = train_log_select_best(
        studies, X_train, y_train, X_test, y_test
    )

    register_best_model(best, catalog, schema, prefix, horizon)

# =========================
# EXECUTE
# =========================
main()
